# ML Baseline Comparison — Contribution 1 (Sealed Test, n = 109)

Non-graph tabular classifiers evaluated under the **same sealed-test protocol** as the GNN,
so their rows are directly comparable to GraphSAGE and feed the same McNemar / DeLong tests.

**Models (ranked by expected relevance):** XGBoost, Random Forest, MLP, Logistic Regression,
SVM-RBF, Gaussian Naive Bayes.

**Protocol.** All 48 features (the BASELINE / all-feature condition, paired to the GraphSAGE
headline). SMOTE and scaling are fit **inside CV folds only**, never on the sealed test and
never before the split — identical leakage discipline to the GNN. Each model is tuned by
inner stratified CV on the training pool, refit on the full pool, and evaluated **once** on
the sealed 109 with no threshold tuning against it. Per-patient `y_true`/`probs`/`preds` are
exported in the shape `statistical-tests.ipynb` already consumes.

**Sealed-split guarantee.** Rather than trust that a re-derived split matches the GNN's, the
notebook loads the stored sealed labels from `per_patient_sealed_predictions_FS.json` and
asserts the reconstructed `y_test` equals them element-for-element; it halts on any mismatch.

**Honest scope.** This is *GNN-on-graph vs standard classifiers on raw routine features*, not a
controlled graph-vs-no-graph ablation: the GNN is transductive (test nodes present during
training) and sees Node2Vec structural embeddings the tabular models cannot. Expect XGBoost or
RF to be competitive — if so, report it plainly; it reframes the contribution rather than
weakening it.

In [1]:
# CELL 1 - INSTALLS
!pip -q install xgboost imbalanced-learn scikit-learn

In [2]:
# CELL 2 - IMPORTS + CONFIG
import json, os, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import (matthews_corrcoef, roc_auc_score, average_precision_score,
                             f1_score, accuracy_score, precision_score, recall_score,
                             cohen_kappa_score, confusion_matrix, make_scorer)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

SEED         = 42
TEST_SIZE    = 0.20
INNER_FOLDS  = 5
N_FEAT_EXPECT= 48
CSV_PATH     = '/kaggle/input/datasets/monamehrun/pcos-cleaned-dataset/pcos_cleaned.csv'
TEST_CSV_PATH= '/kaggle/input/datasets/monamehrun/pcos-test-set/pcos_test_set.csv'   
YTRUE_PATH   = '/kaggle/input/datasets/galibbhai/sealed-predictions/per_patient_sealed_predictions_FS.json'  
OUT          = '/kaggle/working/'
TARGET_CANDIDATES = ['PCOS','PCOS (Y/N)','PCOS_YN','Target','target','Outcome']
DROP_EXTRA   = []  

np.random.seed(SEED)
os.makedirs(OUT, exist_ok=True)

In [3]:
# CELL 3 - LOAD CLEANED + SAVED SEALED-TEST CSVs, ENFORCE FEATURE PARITY
df_full = pd.read_csv(CSV_PATH)
df_test = pd.read_csv(TEST_CSV_PATH)
target = next((c for c in TARGET_CANDIDATES if c in df_test.columns and c in df_full.columns), None)
assert target is not None, f"target not found in both files; test cols={list(df_test.columns)[:6]}"
drop_cols = [target] + [c for c in DROP_EXTRA if c in df_full.columns]
feat_cols = [c for c in df_full.columns if c not in drop_cols]     # cleaned-file column order, minus target
assert len(feat_cols) == N_FEAT_EXPECT, f"{len(feat_cols)} features != BASELINE {N_FEAT_EXPECT} (adjust DROP_EXTRA)"
assert all(c in df_test.columns for c in feat_cols), "feature columns differ between cleaned and test CSV"
print(f"cleaned={len(df_full)} | sealed_test={len(df_test)} | features={len(feat_cols)} | target='{target}'")

cleaned=541 | sealed_test=109 | features=48 | target='PCOS'


In [4]:
# CELL 4 - SPLIT EXACTLY AS THE GNN NOTEBOOKS (NB3 pattern) + IDENTITY ASSERTION
# Training pool = stratified split train-half of the CLEANED file (index-based on labels+seed,
# NOT value-matched). Sealed test = the saved pcos_test_set.csv, loaded directly. This mirrors
# thesis-final-evalution.ipynb Cell 3 and evaluation.ipynb Cell 5 verbatim in logic.
y_full = df_full[target].values.astype(int)
X_full = df_full[feat_cols].values.astype(float)
X_tr, _, y_tr, _ = train_test_split(
    X_full, y_full, test_size=TEST_SIZE, stratify=y_full, random_state=SEED)

y_te = df_test[target].values.astype(int)
X_te = df_test[feat_cols].values.astype(float)

stored = json.load(open(YTRUE_PATH))
y_true_ref = np.asarray(stored['BASELINE']['y_true']).astype(int)
assert len(X_te) == len(y_true_ref) == 109, f"sealed size {len(X_te)} vs {len(y_true_ref)}"
assert int((y_te != y_true_ref).sum()) == 0, (
    "sealed y_test != stored y_true - the test CSV order does not match the GNN's; stop and reconcile")
assert len(X_tr) == len(df_full) - 109, f"train pool {len(X_tr)} != {len(df_full)-109}"
print(f"train pool={len(X_tr)} (pos {int(y_tr.sum())}) | sealed test={len(X_te)} (pos {int(y_te.sum())}) | identity OK")

train pool=432 (pos 141) | sealed test=109 (pos 36) | identity OK


In [5]:
# CELL 5 - MODELS + GRIDS + PIPELINE (in-fold scaling + SMOTE) + MCC SCORER
def pipe(clf):
    return ImbPipeline([('scaler', StandardScaler()),
                        ('smote', SMOTE(random_state=SEED, k_neighbors=5)),
                        ('clf', clf)])

MODELS = {
 'XGBoost': (XGBClassifier(eval_metric='logloss', random_state=SEED, n_jobs=-1, tree_method='hist'),
             {'clf__n_estimators':[200,400], 'clf__max_depth':[3,5],
              'clf__learning_rate':[0.05,0.1], 'clf__subsample':[0.8,1.0]}),
 'RandomForest': (RandomForestClassifier(random_state=SEED, n_jobs=-1),
             {'clf__n_estimators':[200,400], 'clf__max_depth':[None,6,12],
              'clf__min_samples_leaf':[1,3]}),
 'MLP': (MLPClassifier(random_state=SEED, max_iter=500, early_stopping=True),
             {'clf__hidden_layer_sizes':[(64,),(64,32)], 'clf__alpha':[1e-4,1e-3]}),
 'LogReg': (LogisticRegression(max_iter=2000, solver='liblinear', random_state=SEED),
             {'clf__C':[0.01,0.1,1,10], 'clf__penalty':['l1','l2']}),
 'SVM_RBF': (SVC(kernel='rbf', probability=True, random_state=SEED),
             {'clf__C':[1,10], 'clf__gamma':['scale',0.01]}),
 'NaiveBayes': (GaussianNB(), {'clf__var_smoothing':[1e-9,1e-8,1e-7]}),
}
mcc_scorer = make_scorer(matthews_corrcoef)
inner_cv   = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=SEED)

In [6]:
# CELL 6 - METRIC BATTERY (mirrors the GNN table columns)
def metrics(y_true, prob):
    pred=(prob>=0.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_true,pred).ravel()
    return dict(accuracy=accuracy_score(y_true,pred),
                precision=precision_score(y_true,pred,zero_division=0),
                recall=recall_score(y_true,pred),
                f1=f1_score(y_true,pred),
                mcc=matthews_corrcoef(y_true,pred),
                auroc=roc_auc_score(y_true,prob),
                auprc=average_precision_score(y_true,prob),
                sensitivity=tp/(tp+fn) if (tp+fn) else 0.0,
                specificity=tn/(tn+fp) if (tn+fp) else 0.0,
                kappa=cohen_kappa_score(y_true,pred),
                cm=dict(tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp)))

In [7]:
# CELL 7 - TUNE (inner CV) -> REFIT ON POOL -> PREDICT SEALED (incremental save)
results={}; per_patient={}
res_path=f'{OUT}ml_baseline_results.json'; pp_path=f'{OUT}per_patient_ml_baselines.json'
for name,(clf,grid) in MODELS.items():
    gs=GridSearchCV(pipe(clf), grid, scoring=mcc_scorer, cv=inner_cv, n_jobs=-1, refit=True)
    gs.fit(X_tr, y_tr)
    prob=gs.predict_proba(X_te)[:,1]; pred=(prob>=0.5).astype(int)
    m=metrics(y_te, prob)
    results[name]={'best_params':{k.replace('clf__',''):v for k,v in gs.best_params_.items()},
                   'inner_cv_mcc':float(gs.best_score_), **m}
    per_patient[name]={'y_true':y_te.astype(int).tolist(),
                       'probs':prob.astype(float).tolist(),
                       'preds':pred.astype(int).tolist()}
    json.dump(results, open(res_path,'w'), indent=2)      # crash-safe incremental save
    json.dump(per_patient, open(pp_path,'w'))
    print(f"{name:13s} MCC={m['mcc']:.4f} AUROC={m['auroc']:.4f} F1={m['f1']:.4f} "
          f"(inner-CV MCC={gs.best_score_:.4f})")

XGBoost       MCC=0.8122 AUROC=0.9482 F1=0.8732 (inner-CV MCC=0.7602)
RandomForest  MCC=0.8323 AUROC=0.9326 F1=0.8824 (inner-CV MCC=0.7840)
MLP           MCC=0.6912 AUROC=0.9148 F1=0.7945 (inner-CV MCC=0.6891)
LogReg        MCC=0.8148 AUROC=0.9521 F1=0.8767 (inner-CV MCC=0.7581)
SVM_RBF       MCC=0.7685 AUROC=0.9311 F1=0.8406 (inner-CV MCC=0.6994)
NaiveBayes    MCC=0.4358 AUROC=0.8640 F1=0.6408 (inner-CV MCC=0.4073)


In [8]:
# CELL 8 - RESULTS TABLE (ranked by MCC) + SAVE SUMMARY CSV
cols=['mcc','auroc','auprc','f1','accuracy','sensitivity','specificity','kappa']
tab=pd.DataFrame({k:{c:v[c] for c in cols} for k,v in results.items()}).T
tab=tab.sort_values('mcc', ascending=False).round(4)
tab.insert(0,'inner_cv_mcc', [round(results[i]['inner_cv_mcc'],4) for i in tab.index])
print(tab.to_string())
tab.to_csv(f'{OUT}ml_baseline_summary.csv')
# reference line: GraphSAGE sealed anchor for context
print('\n[reference] GraphSAGE sealed MCC=0.8110  AUROC=0.9486')

              inner_cv_mcc     mcc   auroc   auprc      f1  accuracy  sensitivity  specificity   kappa
RandomForest        0.7840  0.8323  0.9326  0.9208  0.8824    0.9266       0.8333       0.9726  0.8293
LogReg              0.7581  0.8148  0.9521  0.9499  0.8767    0.9174       0.8889       0.9315  0.8147
XGBoost             0.7602  0.8122  0.9482  0.9380  0.8732    0.9174       0.8611       0.9452  0.8120
SVM_RBF             0.6994  0.7685  0.9311  0.9130  0.8406    0.8991       0.8056       0.9452  0.7670
MLP                 0.6891  0.6912  0.9148  0.8737  0.7945    0.8624       0.8056       0.8904  0.6911
NaiveBayes          0.4073  0.4358  0.8640  0.7335  0.6408    0.6606       0.9167       0.5342  0.3701

[reference] GraphSAGE sealed MCC=0.8110  AUROC=0.9486


In [9]:
# CELL 9 - ZIP OUTPUTS
import zipfile
zp=f'{OUT}ML_Baselines.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT):
        if f.endswith(('.json','.csv')) and not f.endswith('.zip'):
            z.write(os.path.join(OUT,f), f)
print('zipped ->', zp)
print('per-patient file feeds statistical-tests.ipynb directly:', pp_path)

zipped -> /kaggle/working/ML_Baselines.zip
per-patient file feeds statistical-tests.ipynb directly: /kaggle/working/per_patient_ml_baselines.json
